# StanceEval2026 Kaggle Runner

This notebook is only a thin execution interface. The training, evaluation, and submission logic lives in `src/` and `scripts/`.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import subprocess
import sys
from pathlib import Path

search_roots = [Path.cwd()]
if Path('/kaggle/input').exists():
    search_roots.append(Path('/kaggle/input'))

requirements = None
for root in search_roots:
    for path in root.rglob('requirements.txt'):
        requirements = path
        break
    if requirements is not None:
        break

if requirements is None:
    raise FileNotFoundError('requirements.txt was not found.')

print('Installing from:', requirements)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)])

In [ ]:
import os
import shutil
from pathlib import Path

is_kaggle = Path('/kaggle/working').exists()

def find_project_dir():
    candidates = [Path.cwd()]
    input_root = Path('/kaggle/input')
    if input_root.exists():
        candidates.extend(path.parent for path in input_root.rglob('configs/config.yaml'))
    for candidate in candidates:
        if (candidate / 'configs/config.yaml').exists() and (candidate / 'scripts').exists():
            return candidate.resolve()
    raise FileNotFoundError('Could not find project files. Expected configs/config.yaml and scripts/.')

project_dir = find_project_dir()
if is_kaggle and str(project_dir).startswith('/kaggle/input'):
    work_project = Path('/kaggle/working/stanceeval_project')
    if work_project.exists():
        shutil.rmtree(work_project)
    shutil.copytree(project_dir, work_project)
    project_dir = work_project

os.chdir(project_dir)

os.environ.setdefault('STANCEEVAL_DATA_DIR', str(project_dir))
os.environ.setdefault('STANCEEVAL_OUTPUT_DIR', '/kaggle/working/outputs' if is_kaggle else 'outputs')
os.environ.setdefault('STANCEEVAL_RESULTS_DIR', '/kaggle/working/results' if is_kaggle else 'results')
os.environ.setdefault('STANCEEVAL_BEST_MODEL_DIR', '/kaggle/working/outputs/best_model' if is_kaggle else 'outputs/best_model')

print('Data:', os.environ['STANCEEVAL_DATA_DIR'])
print('Outputs:', os.environ['STANCEEVAL_OUTPUT_DIR'])
print('Results:', os.environ['STANCEEVAL_RESULTS_DIR'])
print('Project:', project_dir)

In [ ]:
!python scripts/run_cls4_experiments.py --config configs/config.yaml --base-model-id aubmindlab/bert-base-arabertv02-twitter

In [ ]:
!python scripts/generate_submissions.py --config configs/config.yaml

In [ ]:
import pandas as pd
from pathlib import Path

results_dir = Path(os.environ['STANCEEVAL_RESULTS_DIR'])
for name in ['experiment_comparison.csv']:
    path = results_dir / name
    if path.exists():
        display(pd.read_csv(path))

print('submission_seen:', results_dir / 'submission_seen.csv')
print('submission_unseen:', results_dir / 'submission_unseen.csv')
print('submission_zip:', results_dir / 'submission_files.zip')